# Kalibrasi LensaLabel v4 di Colab

40 foto kalibrasi, GT final 392 anotasi, KB 240 bahan. Pilih runtime GPU, lalu jalankan sel berurutan. Sel pertama meminta ZIP lensalabel_calibration_v4_bundle.zip yang sudah dibuat di workspace. Produksi tetap memakai closing-marker dan reading-order final. Grid hanya menguji threshold 70–95, tanpa menggunakan 160 foto evaluasi.

Panduan impor notebook dan runtime: [FAQ Colab](https://research.google.com/colaboratory/faq.html). Opsi GPU: [EasyOCR](https://jaided.ai/easyocr/documentation/).

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, platform, shutil, subprocess, sys, zipfile
from google.colab import files

PROJECT = Path("/content/lensalabel_calibration_v4")
BUNDLE_PATH = Path("/content/lensalabel_calibration_v4_bundle.zip")
if not BUNDLE_PATH.exists():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Pilih satu ZIP bundle kalibrasi.")
    BUNDLE_PATH = Path(next(iter(uploaded))).resolve()

PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    for name in archive.namelist():
        if not (PROJECT / name).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError("Path ZIP berada di luar folder proyek.")
    archive.extractall(PROJECT)
manifest = json.loads((PROJECT / "bundle_manifest.json").read_text())
for name, expected in manifest["files_sha256"].items():
    if hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"SHA-256 tidak cocok: {name}")
os.chdir(PROJECT)
with (PROJECT / "evaluation/splits/calibration_set.csv").open(encoding="utf-8-sig", newline="") as source:
    split = list(csv.DictReader(source))
ground_truth = json.loads((PROJECT / "dataset/ground_truth/calibration_ingredients_gt_final.json").read_text())
assert len(split) == 40 and set(ground_truth) == {row["filename"] for row in split}
print(f"Bundle terverifikasi: {len(split)} foto, {manifest['gt_ingredients']} anotasi.")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(PROJECT / "evaluation/requirements-colab.txt")], check=True)

Folder hasil berikut dapat diganti dengan folder Drive jika ingin menyimpan cache antar runtime. Folder Drive harus sudah dipasang sendiri. Pada runtime ini, mengulang sel OCR akan melewati cache yang sudah lengkap.

In [ ]:
import importlib.metadata
import torch
sys.path.insert(0, str(PROJECT / "backend"))
from app.ocr.reader import MAX_OCR_WIDTH
from app.matching.fuzzy_matcher import WEIGHTS
from app.preprocessing.pipeline import preprocess
from app.tokenizer.tokenizer import extract_composition_section

OUTPUT_DIR = PROJECT / "evaluation/results/colab_v4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_DIR / "ocr_cache"
os.environ["EASYOCR_GPU"] = "1" if torch.cuda.is_available() else "0"
assert MAX_OCR_WIDTH == 2000

signature = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
signature_path = OUTPUT_DIR / "input_signature.txt"
if signature_path.exists() and signature_path.read_text().strip() != signature:
    raise ValueError("Folder hasil berisi input berbeda. Gunakan OUTPUT_DIR baru.")
signature_path.write_text(signature + "\n")
runtime = {
    "python": sys.version, "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "EASYOCR_GPU": os.environ["EASYOCR_GPU"], "max_ocr_width": MAX_OCR_WIDTH,
    "weights": WEIGHTS, "thresholds": [70, 75, 80, 85, 90, 95],
    "input_manifest": manifest,
    "versions": {name: importlib.metadata.version(name)
                 for name in ["easyocr", "rapidfuzz", "SQLAlchemy", "torch", "torchvision", "numpy"]},
}
(OUTPUT_DIR / "runtime.json").write_text(json.dumps(runtime, indent=2) + "\n")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
(OUTPUT_DIR / "pip_freeze.txt").write_text(freeze.stdout)
print("GPU:", runtime["gpu"] or "CPU aktif")
print("Folder hasil:", OUTPUT_DIR)

def run_logged(arguments, filename):
    with (OUTPUT_DIR / filename).open("w", encoding="utf-8") as logfile:
        process = subprocess.Popen([sys.executable, *map(str, arguments)], cwd=PROJECT,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in process.stdout:
            print(line, end="")
            logfile.write(line)
            logfile.flush()
        process.wait()
    if process.returncode:
        raise RuntimeError(f"Perintah gagal; baca {filename}.")

OCR dijalankan sekali per foto. Jangan menjalankan grid sebelum seluruh 40 cache lengkap.

In [ ]:
run_logged([
    "evaluation/run_ocr_cache.py",
    "--split-csv", "evaluation/splits/calibration_set.csv",
    "--photos-dir", "dataset/raw_photos",
    "--cache-dir", CACHE_DIR,
], "ocr_log.txt")
missing = [row["filename"] for row in split if not (CACHE_DIR / f"{row['filename']}.json").exists()]
if missing:
    raise ValueError(f"Cache OCR belum lengkap: {missing}")
print("40 cache OCR lengkap.")

In [ ]:
run_logged([
    "evaluation/run_grid_search.py",
    "--split-csv", "evaluation/splits/calibration_set.csv",
    "--cache-dir", CACHE_DIR,
    "--kb-json", "evaluation/kb_snapshot.json",
    "--ground-truth-json", "dataset/ground_truth/calibration_ingredients_gt_final.json",
    "--detail-csv", OUTPUT_DIR / "grid_detail.csv",
    "--summary-csv", OUTPUT_DIR / "grid_summary.csv",
], "grid_log.txt")

In [ ]:
with (OUTPUT_DIR / "grid_summary.csv").open(encoding="utf-8-sig", newline="") as source:
    summaries = list(csv.DictReader(source))
assert len(summaries) == 6 and all(int(row["n_photos"]) == 40 for row in summaries)
ranking = sorted(summaries, key=lambda row: (-float(row["f1"]), -float(row["precision"]), int(row["threshold"])))
print("threshold  precision  recall  f1")
for row in ranking:
    print(f"{row['threshold']:>9}  {float(row['precision']):.4f}     {float(row['recall']):.4f}  {float(row['f1']):.4f}")
selection = {
    "selected_on": "40_photo_calibration_only",
    "tie_break": "highest_f1_then_highest_precision_then_lowest_threshold",
    "best": ranking[0], "weights_unchanged": WEIGHTS,
    "kb_sha256": manifest["files_sha256"]["evaluation/kb_snapshot.json"],
    "final_gt_sha256": manifest["files_sha256"]["dataset/ground_truth/calibration_ingredients_gt_final.json"],
}
(OUTPUT_DIR / "best_threshold.json").write_text(json.dumps(selection, indent=2) + "\n")
print("Threshold produksi belum diubah. Hasil evaluasi 160 foto belum dihitung.")

Unduh ZIP hasil sebelum menutup runtime. Paket memuat 40 cache OCR, CSV detail/summary, log, versi lingkungan, dan best_threshold.json.

In [ ]:
result_zip = shutil.make_archive("/content/lensalabel_calibration_v4_results", "zip", root_dir=OUTPUT_DIR)
files.download(result_zip)